# Augmented RAG with CauseNet + PubMed

This notebook demonstrates how to use the enhanced AugmentedModelSuggester that combines:
- **CauseNet**: Structured causal knowledge base
- **PubMed**: Scientific literature abstracts

The system automatically:
1. Searches CauseNet for matching causal pairs
2. Queries PubMed with intelligent query rewriting
3. Combines both sources into unified retriever
4. Generates LLM response with augmented context

## Setup

In [1]:
import sys
import os
import time
import pandas as pd


# Add the notebook setup path to sys.path
notebook_setup_path = os.path.abspath("../")
sys.path.insert(0, notebook_setup_path)

from notebook_setup import setup_local_pywhyllm
project_root = setup_local_pywhyllm()

📋 Current sys.path before adding project root:
  0: /home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks
  1: /home/moleropa/miniforge3/envs/tfmenv/lib/python311.zip
  2: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11
  3: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/lib-dynload
  4: 
  5: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/site-packages

🎯 Project root to add: /home/moleropa/repositories/master/TFM/pywhyllm
✅ Added local pywhyllm source to Python path: /home/moleropa/repositories/master/TFM/pywhyllm

📋 Updated sys.path after adding project root:
  0: /home/moleropa/repositories/master/TFM/pywhyllm ⭐
  1: /home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks
  2: /home/moleropa/miniforge3/envs/tfmenv/lib/python311.zip
  3: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11
  4: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/lib-dynload
  5: 
  6: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/site-packages
📋 Current sys.p

### Tuebingen dataset   


In [2]:
#df = pd.read_csv('/content/drive/MyDrive/pywhy-llm/pywhyllm/tuebingen_pairs.csv')
df = pd.read_csv('/home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/tuebingen_pairs.csv')


# Modeler LLM

In [3]:
from dotenv import load_dotenv
import guidance
from openai import OpenAI
from portkey_ai import createHeaders

load_dotenv()

# Initialize LLM (using your Azure/OpenAI setup)
azure_model = "gpt-4o-mini"
us_base_url = "https://us.aigw.galileo.roche.com/v1"

portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])

# For guidance (used by SimpleModelSuggester methods)
model = guidance.models.OpenAI(
    azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers
)

# For LangChain (used by query_llm in RAG)
from langchain_openai import ChatOpenAI

langchain_llm = ChatOpenAI(
    model=azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers,
    temperature=0
)

print("✓ Guidance LLM initialized")
print("✓ LangChain LLM initialized")

✓ Guidance LLM initialized
✓ LangChain LLM initialized


testing Augmented 

In [4]:
from pywhyllm.suggesters.augmented_model_suggester_alba import AugmentedModelSuggester


# Use existing CauseNet file (avoids SSL download errors)
#causenet_path = "data/causenet-precision.jsonl.bz2"
causenet_path ="/home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/data/causenet-precision.jsonl.bz2"
# Initialize with both Guidance and LangChain LLMs
suggester = AugmentedModelSuggester(
    llm=model,  # Guidance model for simple suggester methods
    langchain_llm=langchain_llm,  # LangChain model for RAG queries
    pubmed_email="albamaria.molero.perez@example.com",  # Replace with your email
    file_path=causenet_path  # Use existing local file
)

print("✓ AugmentedModelSuggester initialized")
print(f"  CauseNet entries loaded: {len(suggester.causenet_dict)}")

✓ CauseNet found locally at /home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/data/causenet-precision.jsonl.bz2
Loading CauseNet using json
Done loading CauseNet using json
Creating dictionary from CauseNet json data
Done creating dictionary from CauseNet json data
✓ AugmentedModelSuggester initialized
  CauseNet entries loaded: 197806


In [ ]:
from typing import Dict, List, Tuple
llm_output : Dict[str, dict] = {}

# Define parameters for the experiment
temperature = 0.3
num_runs = 1 # Reduced for testing

# Create saved_pairs_info to store ground truth and variable info
saved_pairs_info = {}

# Process only the first 3 rows for testing
#test_df = df.head(1) #or just df all dataset
test_df = df #or just df all dataset
#test_df = df.iloc[9:21]  


# Iterate through each pair and run multiple times
start = time.time()

for pair_number, values in test_df.iterrows():
    pair_id = f"pair{pair_number:04d}"  # Create pair ID like "pair0001", "pair0002", etc.

    var1_value = values['var1'].strip() if isinstance(values['var1'], str) else values['var1']
    var2_value = values['var2'].strip() if isinstance(values['var2'], str) else values['var2']
    ground_truth_value = values['ground_truth']
    if isinstance(ground_truth_value, str):
        ground_truth_value = ground_truth_value.strip().upper()

    saved_pairs_info[pair_id] = {
        "var1": var1_value,
        "var2": var2_value,
        "ground_truth": ground_truth_value,  # columna en tu dataframe
        "truth_ab": int(values['truth_ab']),
        "truth_ba": int(values['truth_ba'])
    }
    
    for n in range(1, num_runs + 1):  # Run 1 to 5
        temp_dict = {}
        
        print(f"Processing {pair_id}, run {n}/{num_runs}")

        temp_dict['llm_ab'] = suggester.suggest_pairwise_relationship(
        variable1=var1_value, 
        variable2=var2_value, 
        use_pubmed=True,
        max_pubmed_papers=5,
        return_prompt=True, 
        confidence_level=True
)
                # )
        temp_dict['llm_ba'] = suggester.suggest_pairwise_relationship(
        variable1=var2_value, 
        variable2=var1_value, 
        use_pubmed=True,
        max_pubmed_papers=5,
        return_prompt=True, 
        confidence_level=True
)
        # Store results with key: (pair_id, temperature, run_number)
        llm_output[(pair_id, temperature, n )] = temp_dict
        
        print(f"  A->B: {temp_dict['llm_ab']}, B->A: {temp_dict['llm_ba']}")

# Calculate latencies after all processing is complete
total_time = time.time() - start
print(f"\nCompleted processing {len(test_df)} pairs with {num_runs} runs each")
print(f"Total execution time: {total_time:.2f}s")

# Calculate latencies
avg_latency_per_pair = total_time / (len(test_df) * num_runs)  # Time per pair (both A->B and B->A) per run
avg_latency_per_run = total_time / (len(test_df) * num_runs * 2)  # Time per individual query (A->B or B->A)

print(f"Average time per pair (A->B + B->A): {avg_latency_per_pair:.2f}s")
print(f"Average time per individual query: {avg_latency_per_run:.2f}s")

Processing pair0000, run 1/1

🔬 Analyzing: Altitude ↔ Temperature

📖 Step 1: Searching CauseNet...


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


📚 Total papers retrieved: 3
   ✓ Retrieved PubMed literature (text length: 6221 chars)

🤖 Step 3: Querying LLM...
📊 Creating combined retriever (text length: 6260 chars)


INFO:backoff:Backing off send_request(...) for 0.7s (requests.exceptions.SSLError: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1016)'))))
INFO:backoff:Backing off send_request(...) for 0.7s (requests.exceptions.SSLError: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1016)'))))
INFO:backoff:Backing off send_request(...) for 0.5s (requests.exceptions.SSLError: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed

   ✓ LLM response generated with RAG augmentation

📋 Step 4: Parsing result...
   ℹ Confidence score: 0.950
   ℹ Strength score: 0.800
   → Causal direction: Horsepower → Acceleration

  A->B: {'result': ['Horsepower', 'Acceleration', "Reasoning:\nHorsepower is a measure of the engine's power output in a vehicle. Acceleration refers to how quickly a vehicle can increase its speed. In physics and engineering, the ability of a vehicle to accelerate is directly influenced by the amount of power (horsepower) the engine can deliver, relative to the vehicle's mass. More horsepower allows a vehicle to overcome inertia and resistance more quickly, resulting in higher acceleration. Conversely, acceleration is a result of the available horsepower and does not itself cause an increase in horsepower. Therefore, the causal direction is from horsepower to acceleration.\n\nFinal answer:\n<answer>A</answer>\n<confidence>0.98</confidence>\n<strength>0.9</strength>"], 'system_prompt': 'You are a helpful

INFO:backoff:Backing off send_request(...) for 2.3s (requests.exceptions.SSLError: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1016)'))))
INFO:backoff:Backing off send_request(...) for 2.3s (requests.exceptions.SSLError: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1016)'))))
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransforme

In [25]:
llm_output

{('pair0000',
  0.3,
  1): {'llm_ab': {'result': ['Altitude',
    'Temperature',
    "Reasoning:\nStep 1: Altitude refers to the height above sea level, while temperature is a measure of thermal energy in the atmosphere.\nStep 2: In physical geography and meteorology, it is well-established that as altitude increases, temperature generally decreases. This is due to the thinning of the atmosphere and lower air pressure at higher elevations, which leads to less heat retention.\nStep 3: Temperature does not influence altitude; altitude is a physical property of the Earth's surface, while temperature is a result of atmospheric conditions.\nStep 4: Therefore, the causal direction is from altitude to temperature: changes in altitude cause changes in temperature, not the other way around.\n\nFinal answer:\n<answer>A</answer>\n<confidence>1.0</confidence>\n<strength>0.9</strength>"],
   'system_prompt': 'You are a helpful assistant for causal reasoning.\n\n    Context: showed positive correlat

### Hallbayes

In [10]:
# Create hallbayes backend using the existing Azure OpenAI client
# We need to bypass the default OpenAI API key requirement
import os
from hallbayes import OpenAIBackend, OpenAIItem, OpenAIPlanner
temp_api_key = os.environ.get("PORTKEY_AZURE_US_API_KEY")

# Create the backend with our Azure model and then replace the client
hallbayes_backend = OpenAIBackend(model=azure_model, api_key=temp_api_key)

azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)


# Replace the default client with our configured Azure OpenAI client 
hallbayes_backend.client = azure_openai_client

planner = OpenAIPlanner(hallbayes_backend, temperature=0.3)


In [36]:
# HallBayes validation function adapted for RAG method (similar to baseline)
def validate_llm_output_with_hallbayes(llm_output, saved_pairs_info, planner, method="with_evidence", threshold=0.10, n_runs=1):
    """
    Validates causal relationships from llm_output using hallbayes
    Only passes: cause, effect variables + retrieved evidence (CauseNet + PubMed)
    
    Args:
        llm_output: dict with keys (pair_id, temperature, run) containing llm_ab and llm_ba results
        saved_pairs_info: dict with pair information (var1, var2, ground_truth)
        planner: configured hallbayes planner
        method: "closed_book" or "with_evidence" 
        threshold: max acceptable hallucination risk
        n_runs: number of validation runs per relationship (for consistency)
    
    Returns:
        dict: hallucination risk results for each query
    """
    print(f"Validating relationships using {method.upper()} method")
    print(f"Threshold: {threshold:.1%}")
    print("=" * 60)
    
    hallucination_risks = {}
    
    # Process each pair and run combination
    for (pair_id, temp, run_num), output in llm_output.items():
        print(f"\nProcessing {pair_id}, run {run_num}")
        
        pair_info = saved_pairs_info[pair_id]
        var1, var2 = pair_info['var1'], pair_info['var2']
        
        # Extract A→B result - only get cause, effect from result list
        ab_result = output['llm_ab']
        ab_result_list = ab_result.get('result', [None, None, ""])
        ab_cause = ab_result_list[0]
        ab_effect = ab_result_list[1]
        # Combine CauseNet + PubMed text as evidence
        ab_causenet = ab_result.get('causenet_text', "")
        ab_pubmed = ab_result.get('pubmed_text', "")
        ab_evidence = f"CauseNet: {ab_causenet}\n\nPubMed: {ab_pubmed}".strip()
        
        # Extract B→A result - only get cause, effect from result list
        ba_result = output['llm_ba']
        ba_result_list = ba_result.get('result', [None, None, ""])
        ba_cause = ba_result_list[0]
        ba_effect = ba_result_list[1]
        # Combine CauseNet + PubMed text as evidence
        ba_causenet = ba_result.get('causenet_text', "")
        ba_pubmed = ba_result.get('pubmed_text', "")
        ba_evidence = f"CauseNet: {ba_causenet}\n\nPubMed: {ba_pubmed}".strip()
        
        # Validate A→B (only if relationship found)
        if ab_cause is not None and ab_effect is not None:
            print(f"  🔍 Validating A→B: LLM claims {ab_cause} → {ab_effect}")
            # Validate what the LLM claims (the correct direction)
            ab_risk = _calculate_single_hallucination_risk(
                ab_cause, ab_effect, ab_evidence, planner, method, threshold, n_runs
            )
        else:
            print(f"  ⊘ A→B: No relationship found")
            ab_risk = {
                'final_valid': False,
                'avg_risk': 1.0,
                'valid_runs': 0,
                'total_runs': 0
            }
        
        # Validate B→A (only if relationship found)
        if ba_cause is not None and ba_effect is not None:
            print(f"  🔍 Validating B→A: Query asks {var2}→{var1}, LLM claims {ba_cause}→{ba_effect}")
            # Validate the QUERY direction (not what LLM answered)
            # This checks if the B→A question itself is valid
            ba_risk = _calculate_single_hallucination_risk(
                var2, var1, ba_evidence, planner, method, threshold, n_runs
            )
        else:
            print(f"  ⊘ B→A: No relationship found")
            ba_risk = {
                'final_valid': False,
                'avg_risk': 1.0,
                'valid_runs': 0,
                'total_runs': 0
            }
        
        # Store results
        key = (pair_id, temp, run_num)
        hallucination_risks[key] = {
            'ab_hallucination_risk': ab_risk['avg_risk'],
            'ab_is_valid': ab_risk['final_valid'],
            'ba_hallucination_risk': ba_risk['avg_risk'], 
            'ba_is_valid': ba_risk['final_valid']
        }
        
        print(f"  📊 A→B: Risk={ab_risk['avg_risk']:.1%}, Valid={ab_risk['final_valid']}")
        print(f"  📊 B→A: Risk={ba_risk['avg_risk']:.1%}, Valid={ba_risk['final_valid']}")
    
    return hallucination_risks


def _calculate_single_hallucination_risk(cause, effect, evidence, planner, method, threshold, n_runs):
    """
    Calculate hallucination risk for a single causal relationship
    Only evaluates: cause → effect claim with retrieved evidence
    Similar to baseline method validation
    """
    # Prepare prompt based on method (simplified - only assess the causal claim)
    if method == "closed_book":
        prompt = f"""
Causal Knowledge Assessment:

Claim: "{cause}" causes "{effect}"

Question: Based on established scientific knowledge, 
is this causal relationship scientifically valid?

Answer: Yes/No with brief justification.
"""
    elif method == "with_evidence":
        prompt = f"""
STRICT Causal Direction Validation with Evidence:

Retrieved Evidence: {evidence}

Claim to validate: "{cause}" causes "{effect}"

Critical Instructions: 
1. Check if the evidence EXPLICITLY supports "{cause}" → "{effect}" as the PRIMARY causal direction
2. Answer NO if the evidence shows the REVERSE direction ("{effect}" → "{cause}") instead
3. Answer NO if the evidence only discusses correlation without establishing "{cause}" as the cause
4. Direction is CRITICAL: These are OPPOSITE claims that cannot both be true:
   - "{cause}" → "{effect}" 
   - "{effect}" → "{cause}"

Question: Does the evidence EXPLICITLY support that "{cause}" causes "{effect}" (and NOT the reverse direction)?

Answer: Yes (ONLY if evidence clearly shows {cause}→{effect} is the PRIMARY direction) or No (if evidence shows the REVERSE or is ambiguous) with justification.
"""
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Run multiple validations for consistency
    run_results = []
    valid_count = 0
    
    for run in range(n_runs):
        try:
            item = OpenAIItem(prompt=prompt, n_samples=3, m=4, skeleton_policy=method)
            print(f"    Run {run + 1}/{n_runs}: Generating HallBayes samples...")
            metrics = planner.run([item], h_star=threshold, isr_threshold=1.0)
            
            if metrics:
                metric = metrics[0]
                # HallBayes decision_answer: True = accept claim, False = reject (hallucination detected)
                is_valid = metric.decision_answer
                risk = metric.roh_bound
                
                # Debug: print raw metric values
                print(f"    [DEBUG] decision_answer={metric.decision_answer}, roh_bound={metric.roh_bound:.3f}")
                
                if is_valid:
                    valid_count += 1
                
                run_results.append({
                    'valid': is_valid,
                    'risk': risk,
                    'run': run + 1
                })
                
                status = "✅ VALID" if is_valid else "❌ REJECT"
                print(f"    Run {run + 1}: {status} (Risk: {risk:.1%})")
            else:
                run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': 'No metrics'})
                print(f"    Run {run + 1}: ⚠️  ERROR (No metrics)")
                
        except Exception as e:
            run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': str(e)})
            print(f"    Run {run + 1}: ⚠️  ERROR ({str(e)[:50]}...)")
    
    # Calculate final decision and average risk
    avg_risk = sum(r.get('risk', 1.0) for r in run_results) / len(run_results) if run_results else 1.0
    
    # Determine final validation (majority vote)
    final_valid = valid_count > (n_runs / 2) if n_runs > 0 else False
    
    return {
        'final_valid': final_valid,
        'avg_risk': avg_risk,
        'valid_runs': valid_count,
        'total_runs': n_runs,
        'runs': run_results
    }


# Run HallBayes validation on all relationships
print("🚀 Starting HallBayes validation for all pairs (RAG method)...")
print(f"Total pairs to validate: {len(llm_output)}")
print()

hallucination_results = validate_llm_output_with_hallbayes(
    llm_output, 
    saved_pairs_info, 
    planner, 
    method="with_evidence",  # Use evidence from CauseNet + PubMed
    threshold=0.10,  # 10% max hallucination risk
    n_runs=1  # Number of validation runs per relationship
)

print("\n" + "="*60)
print("✅ HallBayes validation completed!")
print(f"Total validations performed: {len(hallucination_results)}")
print("="*60)

🚀 Starting HallBayes validation for all pairs (RAG method)...
Total pairs to validate: 1

Validating relationships using WITH_EVIDENCE method
Threshold: 10.0%

Processing pair0000, run 1
  🔍 Validating A→B: LLM claims Altitude → Temperature
    Run 1/1: Generating HallBayes samples...


INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"


    [DEBUG] decision_answer=True, roh_bound=0.000
    Run 1: ✅ VALID (Risk: 0.0%)
  🔍 Validating B→A: Query asks Temperature→Altitude, LLM claims Altitude→Temperature
    Run 1/1: Generating HallBayes samples...


INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://us.aigw.galileo.roche.com/v1/chat/completions "HTTP/1.1 200 OK"


    [DEBUG] decision_answer=True, roh_bound=0.000
    Run 1: ✅ VALID (Risk: 0.0%)
  📊 A→B: Risk=0.0%, Valid=True
  📊 B→A: Risk=0.0%, Valid=True

✅ HallBayes validation completed!
Total validations performed: 1


In [37]:
results : Dict = {}

# Helper utilities to compare LLM output against dataset direction
def _normalize_term(value):
    if isinstance(value, str):
        return value.strip().lower()
    return value

def _infer_direction(result_list, var1, var2):
    if not result_list:
        return "UNKNOWN"
    cause, effect = result_list[0], result_list[1]
    if cause is None or effect is None:
        return "NONE"
    cause_norm = _normalize_term(cause)
    effect_norm = _normalize_term(effect)
    v1_norm = _normalize_term(var1)
    v2_norm = _normalize_term(var2)
    if cause_norm == v1_norm and effect_norm == v2_norm:
        return "R"
    if cause_norm == v2_norm and effect_norm == v1_norm:
        return "L"
    return "UNKNOWN"

for pair_id, info in saved_pairs_info.items():

    av_correct_ab = 0
    av_correct_ba = 0
    
    # Lists to collect confidence, strength scores, and hallucination risks across runs
    confidence_scores_ab = []
    confidence_scores_ba = []
    strength_scores_ab = []
    strength_scores_ba = []
    hallucination_risks_ab = []
    hallucination_risks_ba = []
    validity_ab = []
    validity_ba = []
    direction_runs_ab = []
    direction_runs_ba = []
    
    gt = info['ground_truth'] if isinstance(info['ground_truth'], str) else str(info['ground_truth'])
    gt = gt.strip().upper()

    for i in range(num_runs):
        key = (pair_id, temperature, i+1)
        ab_result = llm_output[key]['llm_ab']
        ba_result = llm_output[key]['llm_ba']
        
        # Get the result list, confidence score, and strength score from the dict
        ab_list = ab_result['result']  # [cause, effect, response]
        ba_list = ba_result['result']  # [cause, effect, response]
        ab_confidence = ab_result.get('confidence_score')
        ba_confidence = ba_result.get('confidence_score')
        ab_strength = ab_result.get('strength_score')
        ba_strength = ba_result.get('strength_score')
        
        # Get hallucination risk data
        hall_data = hallucination_results.get(key, {})
        ab_hall_risk = hall_data.get('ab_hallucination_risk')
        ba_hall_risk = hall_data.get('ba_hallucination_risk')
        ab_valid = hall_data.get('ab_is_valid')
        ba_valid = hall_data.get('ba_is_valid')
        
        # Store confidence and strength scores (only if not None)
        if ab_confidence is not None:
            confidence_scores_ab.append(ab_confidence)
        if ba_confidence is not None:
            confidence_scores_ba.append(ba_confidence)
        if ab_strength is not None:
            strength_scores_ab.append(ab_strength)
        if ba_strength is not None:
            strength_scores_ba.append(ba_strength)
        if ab_hall_risk is not None:
            hallucination_risks_ab.append(ab_hall_risk)
        if ba_hall_risk is not None:
            hallucination_risks_ba.append(ba_hall_risk)
        if ab_valid is not None:
            validity_ab.append(ab_valid)
        if ba_valid is not None:
            validity_ba.append(ba_valid)
        
        pred_ab_direction = _infer_direction(ab_list, info['var1'], info['var2'])
        pred_ba_direction = _infer_direction(ba_list, info['var1'], info['var2'])  
        direction_runs_ab.append(pred_ab_direction)
        direction_runs_ba.append(pred_ba_direction)
        
        # Evaluate correctness for "Does A cause B?"
        # Only count as correct if prediction matches ground truth (no credit for "NONE" or "UNKNOWN")
        if pred_ab_direction == "R" and gt == "R":
            av_correct_ab += 1
        elif pred_ab_direction == "L" and gt == "L":
            av_correct_ab += 1
        # Note: NONE/UNKNOWN are counted as incorrect (no partial credit)
        
        # Evaluate correctness for "Does B cause A?"
        # Only count as correct if prediction matches ground truth
        if pred_ba_direction == "L" and gt == "L":
            av_correct_ba += 1
        elif pred_ba_direction == "R" and gt == "R":
            av_correct_ba += 1
        # Note: NONE/UNKNOWN are counted as incorrect (no partial credit)

    av_correct_ab /= num_runs
    av_correct_ba /= num_runs
    
    # Calculate average confidence scores
    avg_confidence_ab = sum(confidence_scores_ab) / len(confidence_scores_ab) if confidence_scores_ab else None
    avg_confidence_ba = sum(confidence_scores_ba) / len(confidence_scores_ba) if confidence_scores_ba else None
    
    # Calculate average strength scores
    avg_strength_ab = sum(strength_scores_ab) / len(strength_scores_ab) if strength_scores_ab else None
    avg_strength_ba = sum(strength_scores_ba) / len(strength_scores_ba) if strength_scores_ba else None
    
    # Calculate average hallucination risks
    avg_hall_risk_ab = sum(hallucination_risks_ab) / len(hallucination_risks_ab) if hallucination_risks_ab else None
    avg_hall_risk_ba = sum(hallucination_risks_ba) / len(hallucination_risks_ba) if hallucination_risks_ba else None
    
    # Calculate validity rates
    validity_rate_ab = sum(validity_ab) / len(validity_ab) if validity_ab else None
    validity_rate_ba = sum(validity_ba) / len(validity_ba) if validity_ba else None

    temp : Dict = {}

    temp['PairID'] = pair_id
    temp['CorrectACauseB'] = av_correct_ab
    temp['CorrectBCauseA'] = av_correct_ba
    temp['VarA'] = info['var1']
    temp['VarB'] = info['var2']
    temp['GroundTruth'] = gt
    temp['ConfidenceAB'] = avg_confidence_ab
    temp['ConfidenceBA'] = avg_confidence_ba
    temp['StrengthAB'] = avg_strength_ab  # NEW: Average strength score for A->B
    temp['StrengthBA'] = avg_strength_ba  # NEW: Average strength score for B->A
    temp['HallucinationRiskAB'] = avg_hall_risk_ab
    temp['HallucinationRiskBA'] = avg_hall_risk_ba
    temp['ValidityRateAB'] = validity_rate_ab
    temp['ValidityRateBA'] = validity_rate_ba
    temp['PredictedDirectionAB'] = direction_runs_ab
    temp['PredictedDirectionBA'] = direction_runs_ba

    results[pair_id] = temp
    print(results[pair_id])

{'PairID': 'pair0000', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': 'Altitude', 'VarB': 'Temperature', 'GroundTruth': 'R', 'ConfidenceAB': 1.0, 'ConfidenceBA': 0.95, 'StrengthAB': None, 'StrengthBA': None, 'HallucinationRiskAB': 9.999778782798785e-13, 'HallucinationRiskBA': 9.999778782798785e-13, 'ValidityRateAB': 1.0, 'ValidityRateBA': 1.0, 'PredictedDirectionAB': ['R'], 'PredictedDirectionBA': ['R']}


In [38]:
# Calculate accuracy metrics including hallucination risk
accuracy_results = {}

total_pairs = len(results)
sum_ab_accuracy = 0
sum_ba_accuracy = 0
sum_joint_accuracy = 0
sum_ab_confidence = 0
sum_ba_confidence = 0
sum_ab_hall_risk = 0
sum_ba_hall_risk = 0
sum_ab_validity = 0
sum_ba_validity = 0
count_ab_confidence = 0
count_ba_confidence = 0
count_ab_hall_risk = 0
count_ba_hall_risk = 0
count_ab_validity = 0
count_ba_validity = 0


for pair_id, result in results.items():
    # Individual accuracies per pair (these are already averages from multiple runs, can be decimal)
    correct_ab = result['CorrectACauseB']  # This is the mean accuracy (0.0 to 1.0, can be decimal)
    correct_ba = result['CorrectBCauseA']  # This is the mean accuracy (0.0 to 1.0, can be decimal)
    
    # Get confidence scores and hallucination risks
    confidence_ab = result.get('ConfidenceAB')
    confidence_ba = result.get('ConfidenceBA')
    hall_risk_ab = result.get('HallucinationRiskAB')
    hall_risk_ba = result.get('HallucinationRiskBA')
    validity_ab = result.get('ValidityRateAB')
    validity_ba = result.get('ValidityRateBA')
    
    # Joint accuracy: average of both directions (more nuanced approach)
    joint_accuracy = (correct_ab + correct_ba) / 2.0
    
    # Sum for overall statistics (averaging across all pairs)
    sum_ab_accuracy += correct_ab
    sum_ba_accuracy += correct_ba
    sum_joint_accuracy += joint_accuracy
    
    # Sum confidence scores for overall statistics
    if confidence_ab is not None:
        sum_ab_confidence += confidence_ab
        count_ab_confidence += 1
    if confidence_ba is not None:
        sum_ba_confidence += confidence_ba
        count_ba_confidence += 1
        
    # Sum hallucination risks for overall statistics
    if hall_risk_ab is not None:
        sum_ab_hall_risk += hall_risk_ab
        count_ab_hall_risk += 1
    if hall_risk_ba is not None:
        sum_ba_hall_risk += hall_risk_ba
        count_ba_hall_risk += 1
        
    # Sum validity rates for overall statistics
    if validity_ab is not None:
        sum_ab_validity += validity_ab
        count_ab_validity += 1
    if validity_ba is not None:
        sum_ba_validity += validity_ba
        count_ba_validity += 1
    
    # STORING INDIVIDUAL
    # Store individual pair results (only decimal values, no redundant percentages)
    accuracy_results[pair_id] = {
        'PairID': pair_id,
        'VarA': result['VarA'],
        'VarB': result['VarB'],
        'GroundTruth': result['GroundTruth'],
        'AccuracyAB': correct_ab,  # Decimal value (0.0 to 1.0)
        'AccuracyBA': correct_ba,  # Decimal value (0.0 to 1.0)
        'JointAccuracy': joint_accuracy,  # Average of both directions
        'ConfidenceAB': confidence_ab,  # Confidence score for A->B
        'ConfidenceBA': confidence_ba,   # Confidence score for B->A
        'HallucinationRiskAB': hall_risk_ab,  # Hallucination risk for A->B
        'HallucinationRiskBA': hall_risk_ba,  # Hallucination risk for B->A
        'ValidityRateAB': validity_ab,  # HallBayes validity rate for A->B
        'ValidityRateBA': validity_ba   # HallBayes validity rate for B->A
    }
    
    pred_dirs_ab = result.get('PredictedDirectionAB', [])
    pred_dirs_ba = result.get('PredictedDirectionBA', [])
    print(f"Pair {pair_id}: {result['VarA']} -> {result['VarB']}")
    print(f"  Ground Truth: {result['GroundTruth']}")
    print(f"  A→B Accuracy: {correct_ab:.3f}, Confidence: {f'{confidence_ab:.3f}' if confidence_ab is not None else 'N/A'}")
    print(f"  B→A Accuracy: {correct_ba:.3f}, Confidence: {f'{confidence_ba:.3f}' if confidence_ba is not None else 'N/A'}")
    print(f"  Predicted directions AB runs: {pred_dirs_ab}")
    print(f"  Predicted directions BA runs: {pred_dirs_ba}")
    print(f"  Joint Accuracy (avg): {joint_accuracy:.3f}")
    print()


# Overall accuracy statistics (averages across all pairs)
overall_ab_accuracy = sum_ab_accuracy / total_pairs
overall_ba_accuracy = sum_ba_accuracy / total_pairs
overall_joint_accuracy = sum_joint_accuracy / total_pairs

# Overall confidence statistics
overall_ab_confidence = sum_ab_confidence / count_ab_confidence if count_ab_confidence > 0 else None
overall_ba_confidence = sum_ba_confidence / count_ba_confidence if count_ba_confidence > 0 else None

# Overall hallucination risk statistics
overall_ab_hall_risk = sum_ab_hall_risk / count_ab_hall_risk if count_ab_hall_risk > 0 else None
overall_ba_hall_risk = sum_ba_hall_risk / count_ba_hall_risk if count_ba_hall_risk > 0 else None

# Overall validity statistics
overall_ab_validity = sum_ab_validity / count_ab_validity if count_ab_validity > 0 else None
overall_ba_validity = sum_ba_validity / count_ba_validity if count_ba_validity > 0 else None

print("=== OVERALL ACCURACY STATISTICS ===")
print(f"Total pairs processed: {total_pairs}")
print(f"\nMEAN ACCURACIES (across all pairs):")
print(f"A→B Mean Accuracy: {overall_ab_accuracy:.3f}")
print(f"B→A Mean Accuracy: {overall_ba_accuracy:.3f}")
print(f"Joint Mean Accuracy: {overall_joint_accuracy:.3f}")
print(f"\nMEAN CONFIDENCE SCORES:")
print(f"A→B Mean Confidence: {f'{overall_ab_confidence:.3f}' if overall_ab_confidence is not None else 'N/A'}")
print(f"B→A Mean Confidence: {f'{overall_ba_confidence:.3f}' if overall_ba_confidence is not None else 'N/A'}")
print(f"\nMEAN HALLUCINATION RISKS:")
print(f"A→B Mean Hall Risk: {f'{overall_ab_hall_risk:.1%}' if overall_ab_hall_risk is not None else 'N/A'}")
print(f"B→A Mean Hall Risk: {f'{overall_ba_hall_risk:.1%}' if overall_ba_hall_risk is not None else 'N/A'}")
print(f"\nMEAN VALIDITY RATES:")
print(f"A→B Mean Validity: {f'{overall_ab_validity:.3f}' if overall_ab_validity is not None else 'N/A'}")
print(f"B→A Mean Validity: {f'{overall_ba_validity:.3f}' if overall_ba_validity is not None else 'N/A'}")
print(f"\nLATENCY METRICS:")
print(f"Average time per pair (both directions): {avg_latency_per_pair:.2f}s")
print(f"Average time per query: {avg_latency_per_run:.2f}s")

# Summary statistics 
accuracy_summary = {
    'total_pairs': total_pairs,
    'ab_mean_accuracy': overall_ab_accuracy,
    'ba_mean_accuracy': overall_ba_accuracy,
    'joint_mean_accuracy': overall_joint_accuracy,
    'ab_mean_confidence': overall_ab_confidence,
    'ba_mean_confidence': overall_ba_confidence,
    'ab_mean_hallucination_risk': overall_ab_hall_risk,
    'ba_mean_hallucination_risk': overall_ba_hall_risk,
    'ab_mean_validity': overall_ab_validity,
    'ba_mean_validity': overall_ba_validity,
    'avg_latency_per_pair_seconds': round(avg_latency_per_pair, 2),
    'avg_latency_per_query_seconds': round(avg_latency_per_run, 2),
    'total_execution_time_seconds': round(total_time, 2)
}

Pair pair0000: Altitude -> Temperature
  Ground Truth: R
  A→B Accuracy: 1.000, Confidence: 1.000
  B→A Accuracy: 1.000, Confidence: 0.950
  Predicted directions AB runs: ['R']
  Predicted directions BA runs: ['R']
  Joint Accuracy (avg): 1.000

=== OVERALL ACCURACY STATISTICS ===
Total pairs processed: 1

MEAN ACCURACIES (across all pairs):
A→B Mean Accuracy: 1.000
B→A Mean Accuracy: 1.000
Joint Mean Accuracy: 1.000

MEAN CONFIDENCE SCORES:
A→B Mean Confidence: 1.000
B→A Mean Confidence: 0.950

MEAN HALLUCINATION RISKS:
A→B Mean Hall Risk: 0.0%
B→A Mean Hall Risk: 0.0%

MEAN VALIDITY RATES:
A→B Mean Validity: 1.000
B→A Mean Validity: 1.000

LATENCY METRICS:
Average time per pair (both directions): 27.16s
Average time per query: 13.58s


### Export output

In [ ]:
# Save accuracy results to CSV including hallucination risk metrics
import csv

# CSV file for detailed accuracy results (now includes confidence scores and hallucination risks)
accuracy_csv_file = "alba_accuracy_results_m2_all_pairs_hallbayes.csv"

# Define headers for detailed accuracy results (includes confidence and hallucination risk)
accuracy_header = [
    "PairID", "VarA", "VarB", "GroundTruth", 
    "AccuracyAB", "AccuracyBA", "JointAccuracy",
    "ConfidenceAB", "ConfidenceBA", 
    "HallucinationRiskAB", "HallucinationRiskBA",
    "ValidityRateAB", "ValidityRateBA"
]

# Write detailed accuracy results
with open(accuracy_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=accuracy_header)
    writer.writeheader()
    for pair_id, values in accuracy_results.items():
        writer.writerow(values)

print(f"Detailed accuracy CSV file '{accuracy_csv_file}' has been created.")

# CSV file for summary statistics
summary_csv_file = "alba_accuracy_summary_method2_rag_all_pairs_hallbayes.csv"

# Write summary statistics
with open(summary_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(accuracy_summary.keys()))
    writer.writeheader()
    writer.writerow(accuracy_summary)

print(f"Summary accuracy CSV file '{summary_csv_file}' has been created.")

# Display final summary table (clean format with confidence and hallucination risk)
print("\n=== FINAL SUMMARY TABLE ===")
print(f"{'Metric':<25} {'Mean Accuracy':<15} {'Mean Confidence':<18} {'Mean Hall Risk':<15} {'Mean Validity':<15}")
print("-" * 105)
ab_conf_str = f"{accuracy_summary['ab_mean_confidence']:.3f}" if accuracy_summary['ab_mean_confidence'] is not None else 'N/A'
ba_conf_str = f"{accuracy_summary['ba_mean_confidence']:.3f}" if accuracy_summary['ba_mean_confidence'] is not None else 'N/A'
ab_hall_str = f"{accuracy_summary['ab_mean_hallucination_risk']:.1%}" if accuracy_summary['ab_mean_hallucination_risk'] is not None else 'N/A'
ba_hall_str = f"{accuracy_summary['ba_mean_hallucination_risk']:.1%}" if accuracy_summary['ba_mean_hallucination_risk'] is not None else 'N/A'
ab_valid_str = f"{accuracy_summary['ab_mean_validity']:.3f}" if accuracy_summary['ab_mean_validity'] is not None else 'N/A'
ba_valid_str = f"{accuracy_summary['ba_mean_validity']:.3f}" if accuracy_summary['ba_mean_validity'] is not None else 'N/A'

print(f"{'A→B':<25} {accuracy_summary['ab_mean_accuracy']:<15.3f} {ab_conf_str:<18} {ab_hall_str:<15} {ab_valid_str:<15}")
print(f"{'B→A':<25} {accuracy_summary['ba_mean_accuracy']:<15.3f} {ba_conf_str:<18} {ba_hall_str:<15} {ba_valid_str:<15}")
print(f"{'Joint Accuracy':<25} {accuracy_summary['joint_mean_accuracy']:<15.3f} {'N/A':<18} {'N/A':<15} {'N/A':<15}")
print(f"{'Total Pairs':<25} {accuracy_summary['total_pairs']:<15} {'N/A':<18} {'N/A':<15} {'N/A':<15}")

print(f"\n=== HALLUCINATION RISK ANALYSIS ===")
if accuracy_summary['ab_mean_hallucination_risk'] is not None:
    print(f"Low risk pairs (A→B < 20%): {sum(1 for r in accuracy_results.values() if r.get('HallucinationRiskAB', 1) < 0.20)}")
    print(f"Medium risk pairs (A→B 20-50%): {sum(1 for r in accuracy_results.values() if 0.20 <= r.get('HallucinationRiskAB', 1) < 0.50)}")
    print(f"High risk pairs (A→B >= 50%): {sum(1 for r in accuracy_results.values() if r.get('HallucinationRiskAB', 1) >= 0.50)}")
if accuracy_summary['ba_mean_hallucination_risk'] is not None:
    print(f"Low risk pairs (B→A < 20%): {sum(1 for r in accuracy_results.values() if r.get('HallucinationRiskBA', 1) < 0.20)}")
    print(f"Medium risk pairs (B→A 20-50%): {sum(1 for r in accuracy_results.values() if 0.20 <= r.get('HallucinationRiskBA', 1) < 0.50)}")
    print(f"High risk pairs (B→A >= 50%): {sum(1 for r in accuracy_results.values() if r.get('HallucinationRiskBA', 1) >= 0.50)}")

In [ ]:
# # Debug: Check actual hallucination risk values
# print("=== DEBUG: Raw Hallucination Risk Values ===")
# for pair_id, result in results.items():
#     hall_risk_ab = result.get('HallucinationRiskAB')
#     hall_risk_ba = result.get('HallucinationRiskBA')
#     print(f"\n{pair_id}:")
#     print(f"  AB Risk (raw): {hall_risk_ab}")
#     print(f"  BA Risk (raw): {hall_risk_ba}")
#     print(f"  AB Risk (scientific): {hall_risk_ab:.2e}" if hall_risk_ab else "  AB Risk: None")
#     print(f"  BA Risk (scientific): {hall_risk_ba:.2e}" if hall_risk_ba else "  BA Risk: None")
#     print(f"  AB Risk (percentage): {hall_risk_ab:.10%}" if hall_risk_ab else "  AB Risk: None")
#     print(f"  BA Risk (percentage): {hall_risk_ba:.10%}" if hall_risk_ba else "  BA Risk: None")

# print("\n=== Overall Statistics ===")
# print(f"Mean AB Risk (raw): {overall_ab_hall_risk}")
# print(f"Mean BA Risk (raw): {overall_ba_hall_risk}")
# print(f"Mean AB Risk (scientific): {overall_ab_hall_risk:.2e}" if overall_ab_hall_risk else "Mean AB Risk: None")
# print(f"Mean BA Risk (scientific): {overall_ba_hall_risk:.2e}" if overall_ba_hall_risk else "Mean BA Risk: None")
# print(f"Mean AB Risk (10 decimals): {overall_ab_hall_risk:.10%}" if overall_ab_hall_risk else "Mean AB Risk: None")
# print(f"Mean BA Risk (10 decimals): {overall_ba_hall_risk:.10%}" if overall_ba_hall_risk else "Mean BA Risk: None")

=== DEBUG: Raw Hallucination Risk Values ===

pair0000:
  AB Risk (raw): 9.999778782798785e-13
  BA Risk (raw): 9.999778782798785e-13
  AB Risk (scientific): 1.00e-12
  BA Risk (scientific): 1.00e-12
  AB Risk (percentage): 0.0000000001%
  BA Risk (percentage): 0.0000000001%

=== Overall Statistics ===
Mean AB Risk (raw): 9.999778782798785e-13
Mean BA Risk (raw): 9.999778782798785e-13
Mean AB Risk (scientific): 1.00e-12
Mean BA Risk (scientific): 1.00e-12
Mean AB Risk (10 decimals): 0.0000000001%
Mean BA Risk (10 decimals): 0.0000000001%
